# SAT Crédito — Evaluación del Modelo de Survival Analysis

**Sistema de Alerta Temprana para Mora en Créditos de Consumo**

Este notebook evalúa el desempeño del modelo **CoxPHFitter (lifelines)** entrenado para
predecir el tiempo hasta el primer incumplimiento crediticio.

Estructura del análisis:
1. Métricas globales (C-index de Harrell, Brier Score)
2. Curvas de Kaplan-Meier
3. Hazard Ratios — coeficientes del modelo de Cox
4. Discriminación: distribución de scores de riesgo
5. Calibración — Brier Score por horizonte
6. Importancia de variables
7. Distribución y eficacia de las alertas
8. Curvas Kaplan-Meier por nivel de alerta
9. Comparación train vs test — sobreajuste
10. Reporte final

> **Marco teórico**: el modelo de Cox estima `h(t|X) = h₀(t) · exp(Xβ)`. La función de
> supervivencia `S(t|X) = P(no incumplir hasta t)` es la salida central del modelo.
> El C-index de Harrell mide la concordancia entre el ranking de riesgo predicho y el
> orden temporal observado (análogo del AUC-ROC para datos de supervivencia).

In [ ]:
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 13

VERDE    = '#2ECC71'
AMARILLO = '#F39C12'
ROJO     = '#E74C3C'
AZUL     = '#2980B9'
MORADO   = '#8E44AD'
GRIS     = '#95A5A6'


In [ ]:
# ── Carga de artefactos y datos ────────────────────────────────────────────
features = pd.read_csv('../data/processed/features.csv')
alertas  = pd.read_csv('../data/processed/alertas_sat.csv')
fi_df    = pd.read_csv('../data/processed/feature_importance.csv')

with open('../src/modelo/modelo_sat.pkl', 'rb') as f:
    artifacts = pickle.load(f)

with open('../src/modelo/metricas.json') as f:
    metricas = json.load(f)

preprocessor  = artifacts['preprocessor']
cph           = artifacts['cox_model']
feature_names = artifacts['feature_names']

print('Modelo:', metricas['modelo'])
print(f'C-index (test):  {metricas["c_index_test"]:.4f}')
print(f'C-index (train): {metricas["c_index_train"]:.4f}')
print(f'Overfit gap:     {metricas["overfit_gap"]:.4f}')
features.head(3)


In [ ]:
# Reproducir el split 80/20 del entrenamiento
DURATION_COL = 'tiempo_evento'
EVENT_COL    = 'evento'
EXCLUIR = ['id_solicitud', 'periodo_desembolso', 'nombre_departamento_particular',
           'atraso_30', 'atraso_60', 'atraso_90', DURATION_COL, EVENT_COL]

X       = features.drop(columns=EXCLUIR, errors='ignore').select_dtypes(include=[np.number])
X       = X.reindex(columns=feature_names, fill_value=0)
y_time  = features[DURATION_COL]
y_event = features[EVENT_COL]

X_train, X_test, yt_train, yt_test, ye_train, ye_test = train_test_split(
    X, y_time, y_event, test_size=0.20, stratify=y_event, random_state=42
)

X_test_sc  = pd.DataFrame(preprocessor.transform(X_test),  columns=feature_names)
X_train_sc = pd.DataFrame(preprocessor.transform(X_train), columns=feature_names)

risk_test  = cph.predict_partial_hazard(X_test_sc).values
risk_train = cph.predict_partial_hazard(X_train_sc).values

surv_test  = cph.predict_survival_function(X_test_sc, times=[1, 2, 3])
s1_test    = surv_test.loc[1].values   # S(1): P(sobrevivir 1 mes)
prob_mora  = 1 - s1_test               # P(mora en T=1)

umbral_alto  = metricas['umbral_alerta_alta']
umbral_medio = metricas['umbral_alerta_media']

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Eventos en test: {ye_test.sum():,} ({ye_test.mean()*100:.1f}%)')


---
## 1. Métricas Globales del Modelo de Supervivencia


In [ ]:
c_test  = concordance_index(yt_test,  -risk_test,  ye_test)
c_train = concordance_index(yt_train, -risk_train, ye_train)

resumen = pd.DataFrame([
    {'Métrica': 'C-index (test)',      'Valor': f'{c_test:.4f}',
     'Interpretación': '> 0.85 = excelente discriminación'},
    {'Métrica': 'C-index (train)',     'Valor': f'{c_train:.4f}',
     'Interpretación': 'Brecha < 0.05 = sin sobreajuste'},
    {'Métrica': 'Overfit gap',         'Valor': f'{metricas["overfit_gap"]:.4f}',
     'Interpretación': '< 0.05 = modelo estable'},
    {'Métrica': 'Brier Score (t=1)',   'Valor': f'{metricas["brier_t1"]:.4f}',
     'Interpretación': '0 = perfecto, 0.25 = aleatorio'},
    {'Métrica': 'S(1) mediana',        'Valor': f'{metricas["s1_mediana_test"]:.4f}',
     'Interpretación': 'P(sobrevivir 30 días) en el cliente mediano'},
    {'Métrica': 'P(mora) mediana',     'Valor': f'{1-metricas["s1_mediana_test"]:.4f}',
     'Interpretación': 'P(incumplir) en el cliente mediano'},
    {'Métrica': 'Umbral ALTO',         'Valor': f'{umbral_alto:.4f}',
     'Interpretación': 'P(mora) ≥ umbral → alerta ALTO'},
    {'Métrica': 'Umbral MEDIO',        'Valor': f'{umbral_medio:.4f}',
     'Interpretación': 'P(mora) ≥ umbral → alerta MEDIO'},
])

resumen.style\
    .set_table_styles([{'selector': 'th',
                        'props': [('background-color', '#2980B9'), ('color', 'white')]}])\
    .hide(axis='index')


---
## 2. Curvas de Kaplan-Meier

La estimación de Kaplan-Meier muestra la función de supervivencia S(t) sin asumir
ninguna forma paramétrica. Es la referencia no paramétrica del análisis de supervivencia.

In [ ]:
# Kaplan-Meier global + por grupos de riesgo (terciles del score)
kmf = KaplanMeierFitter()

# Crear grupos de riesgo por tercil del hazard score en test
q33 = np.percentile(risk_test, 33)
q67 = np.percentile(risk_test, 67)

grupo = np.where(risk_test <= q33, 'Bajo riesgo',
        np.where(risk_test <= q67, 'Riesgo medio', 'Alto riesgo'))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# — KM global —
kmf.fit(yt_test, ye_test, label='Todos los clientes')
kmf.plot_survival_function(ax=axes[0], color=AZUL, linewidth=2.5,
                            ci_show=True, ci_alpha=0.15)
axes[0].set_title('Función de Supervivencia Global (KM)', fontweight='bold')
axes[0].set_xlabel('Tiempo (meses)')
axes[0].set_ylabel('S(t) — P(no incumplir hasta t)')
axes[0].set_xlim([0, 40])
axes[0].set_ylim([0, 1.05])

# — KM por grupo de riesgo —
colors_g = {'Alto riesgo': ROJO, 'Riesgo medio': AMARILLO, 'Bajo riesgo': VERDE}
for g, color in colors_g.items():
    mask = grupo == g
    kmf.fit(yt_test[mask], ye_test[mask], label=g)
    kmf.plot_survival_function(ax=axes[1], color=color, linewidth=2.5,
                               ci_show=True, ci_alpha=0.1)

axes[1].set_title('Curvas KM por Grupo de Riesgo (terciles score Cox)',
                  fontweight='bold')
axes[1].set_xlabel('Tiempo (meses)')
axes[1].set_ylabel('S(t)')
axes[1].set_xlim([0, 40])
axes[1].set_ylim([0, 1.05])

# Test log-rank alto vs bajo
mask_a = grupo == 'Alto riesgo'
mask_b = grupo == 'Bajo riesgo'
res = logrank_test(yt_test[mask_a], yt_test[mask_b],
                   ye_test[mask_a], ye_test[mask_b])
axes[1].set_title(f'KM por Grupo de Riesgo\n(log-rank p={res.p_value:.2e})',
                  fontweight='bold')

plt.suptitle('Estimación de Kaplan-Meier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Test log-rank (alto vs bajo riesgo): p = {res.p_value:.2e}')
print('Separación estadísticamente significativa' if res.p_value < 0.05 else 'Sin separación significativa')


---
## 3. Hazard Ratios — Coeficientes del Modelo de Cox

El modelo de Cox estima un coeficiente β por variable. El **Hazard Ratio (HR) = exp(β)**:
- HR > 1: la variable **aumenta** el riesgo (hazard)
- HR < 1: la variable **reduce** el riesgo
- HR = 1: sin efecto

Los coeficientes están regularizados (L2, penalizer=0.10).

In [ ]:
# Forest plot de hazard ratios — Top 20 variables por |coef|
summary = cph.summary.copy()
summary = summary.sort_values('coef', key=abs, ascending=False).head(20)
summary = summary.sort_values('coef')   # orden para el plot

fig, ax = plt.subplots(figsize=(12, 8))

y_pos = range(len(summary))
colors_hr = [ROJO if v > 0 else AZUL for v in summary['coef']]

ax.barh(y_pos, summary['coef'], color=colors_hr, alpha=0.75,
        edgecolor='white', height=0.6)

# IC 95%
ci_low  = summary['coef lower 95%']
ci_high = summary['coef upper 95%']
ax.errorbar(
    summary['coef'], y_pos,
    xerr=[summary['coef'] - ci_low, ci_high - summary['coef']],
    fmt='none', color='black', linewidth=1.2, capsize=3
)

# Línea de referencia HR=1 (coef=0)
ax.axvline(0, color='black', linewidth=1.2, linestyle='--')

# Etiquetas HR en el margen
for i, (_, row) in enumerate(summary.iterrows()):
    hr = np.exp(row['coef'])
    ax.text(summary['coef'].max() * 1.05, i,
            f'HR={hr:.3f}', va='center', fontsize=8)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(summary.index, fontsize=9)
ax.set_xlabel('Coeficiente β (log-hazard ratio)\n← menor riesgo  |  mayor riesgo →')
ax.set_title('Hazard Ratios — Modelo de Cox (Top 20 por |β|)\n'
             'barras rojas: ↑ riesgo   |   barras azules: ↓ riesgo',
             fontweight='bold')

# Leyenda de significancia
sig = (summary['p'] < 0.05).sum()
ax.set_title(ax.get_title() + f'\n{sig}/{len(summary)} variables con p < 0.05', fontsize=11)

plt.tight_layout()
plt.show()

print('\nTop 10 por Hazard Ratio:')
print(summary.sort_values('coef', ascending=False)[['coef','exp(coef)','p']].head(10).to_string())


---
## 4. Discriminación: Distribución de Scores de Riesgo


In [ ]:
# Distribución del score de riesgo relativo (partial hazard = exp(Xβ))
# en clientes que incumplieron vs censurados
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# — Partial hazard (log-scale) por clase —
for val, color, label in [(0, AZUL, 'Censurado (no incumplió)'),
                           (1, ROJO, 'Incumplió (evento=1)')]:
    scores = np.log(risk_test[ye_test.values == val] + 1e-9)
    axes[0].hist(scores, bins=40, alpha=0.55, color=color, label=label,
                 density=True, edgecolor='none')
    sns.kdeplot(scores, ax=axes[0], color=color, linewidth=2)

axes[0].set_xlabel('log(partial hazard) = Xβ')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Score de Riesgo (log partial hazard) por Clase', fontweight='bold')
axes[0].legend()

# — P(mora 30d) por clase —
for val, color, label in [(0, AZUL, 'Censurado'), (1, ROJO, 'Incumplió')]:
    pm = prob_mora[ye_test.values == val]
    axes[1].hist(pm, bins=40, alpha=0.55, color=color, label=label,
                 density=True, edgecolor='none')
    sns.kdeplot(pm, ax=axes[1], color=color, linewidth=2)

axes[1].axvline(umbral_alto,  color='black',  linestyle='--', linewidth=1.8,
                label=f'Umbral ALTO ({umbral_alto:.3f})')
axes[1].axvline(umbral_medio, color=GRIS,     linestyle=':',  linewidth=1.8,
                label=f'Umbral MEDIO ({umbral_medio:.3f})')

# Zonas de alerta
axes[1].axvspan(0,            umbral_medio, alpha=0.05, color=VERDE)
axes[1].axvspan(umbral_medio, umbral_alto,  alpha=0.05, color=AMARILLO)
axes[1].axvspan(umbral_alto,  1.0,          alpha=0.05, color=ROJO)

axes[1].set_xlabel('P(mora ≤ 30 días) = 1 - S(1|X)')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Probabilidad de Mora a 30 Días por Clase', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Discriminación del Modelo de Cox', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# AUC-ROC aproximado sobre P(mora) — para referencia con modelo previo
from sklearn.metrics import roc_auc_score
auc_approx = roc_auc_score(ye_test, prob_mora)
print(f'AUC-ROC aproximado sobre P(mora 30d): {auc_approx:.4f}  (referencia, no métrica principal)')
print(f'C-index de Harrell (métrica survival): {c_test:.4f}')


---
## 5. Calibración — Brier Score

El **Brier Score** mide la calibración de las probabilidades de supervivencia:
BS(t) = E[(I(T≤t) − (1−S(t|X)))²]. Valores menores = mejor calibración.
Referencia: 0 = perfecto, 0.25 = modelo aleatorio (sin información).

In [ ]:
# Brier Score en t=1, 2, 3
brier_vals = [metricas['brier_t1'], metricas['brier_t2'], metricas['brier_t3']]
horizontes = ['t=1 (30 días)', 't=2 (60 días)', 't=3 (90 días)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Brier Score por horizonte —
colors_bs = [VERDE if b < 0.10 else AMARILLO if b < 0.20 else ROJO for b in brier_vals]
bars = axes[0].bar(horizontes, brier_vals, color=colors_bs, width=0.4, edgecolor='white')
axes[0].axhline(0.25, color='gray', linestyle='--', linewidth=1.5, label='Aleatorio (0.25)')
axes[0].axhline(0.10, color=VERDE,  linestyle='--', linewidth=1.5, label='Buena calibración (0.10)')
for bar, val in zip(bars, brier_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_ylabel('Brier Score')
axes[0].set_ylim([0, 0.30])
axes[0].set_title('Brier Score por Horizonte Temporal', fontweight='bold')
axes[0].legend()

# — Calibración: P(mora) predicha vs observada (deciles) —
df_cal = pd.DataFrame({'prob_mora': prob_mora, 'evento': ye_test.values})
df_cal['decil'] = pd.qcut(df_cal['prob_mora'], 10, duplicates='drop')
cal_data = df_cal.groupby('decil', observed=True).agg(
    pred_mean=('prob_mora', 'mean'),
    obs_mean=('evento', 'mean')
).reset_index()

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Calibración perfecta')
axes[1].scatter(cal_data['pred_mean'], cal_data['obs_mean'],
                color=AZUL, s=80, zorder=5, label='Deciles observados')
axes[1].plot(cal_data['pred_mean'], cal_data['obs_mean'],
             color=AZUL, linewidth=1.5, alpha=0.6)
axes[1].set_xlabel('P(mora) media predicha por decil')
axes[1].set_ylabel('Tasa de mora real observada')
axes[1].set_title('Curva de Calibración (deciles de P(mora))', fontweight='bold')
axes[1].legend()
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

plt.suptitle('Calibración del Modelo de Cox', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 6. Importancia de Variables

La importancia en el modelo de Cox se mide por el valor absoluto del coeficiente β
(normalizado). Un HR > 1 indica que la variable aumenta el riesgo; HR < 1 lo reduce.

In [ ]:
fi_top = fi_df.head(20).copy().sort_values('importancia_abs')

# Colores por dirección del efecto
colors_fi = [ROJO if v > 0 else AZUL for v in fi_top['coef']]

fig, axes = plt.subplots(1, 2, figsize=(16, 8), gridspec_kw={'width_ratios': [3, 1]})

bars = axes[0].barh(fi_top['feature'], fi_top['importancia_pct'],
                    color=colors_fi, edgecolor='white', height=0.65, alpha=0.85)
for bar, row in zip(bars, fi_top.itertuples()):
    hr = np.exp(row.coef)
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'HR={hr:.3f}', va='center', fontsize=8)

axes[0].set_xlabel('Importancia relativa (|β| normalizado, %)')
axes[0].set_title('Top 20 Variables — Importancia en Modelo de Cox\n'
                  '(rojo = ↑ riesgo, azul = ↓ riesgo)', fontweight='bold')

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=ROJO, label='Aumenta riesgo (HR > 1)'),
              Patch(facecolor=AZUL, label='Reduce riesgo (HR < 1)')]
axes[1].legend(handles=legend_els, loc='center', frameon=True,
               title='Dirección del efecto', fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
fi_sorted = fi_df.sort_values('importancia_pct', ascending=False).reset_index(drop=True)
fi_sorted['importancia_acum'] = fi_sorted['importancia_pct'].cumsum()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

colores_pareto = [ROJO if v > 0 else AZUL for v in fi_sorted['coef']]
ax1.bar(range(len(fi_sorted)), fi_sorted['importancia_pct'],
        color=colores_pareto, alpha=0.7, label='Importancia individual (%)')
ax2.plot(range(len(fi_sorted)), fi_sorted['importancia_acum'],
         color='black', linewidth=2.5, marker='.', markersize=5)

for nivel, color in [(80, AMARILLO), (95, GRIS)]:
    idx_n = (fi_sorted['importancia_acum'] >= nivel).idxmax()
    ax2.axhline(nivel, linestyle='--', color=color, linewidth=1.3)
    ax2.axvline(idx_n, linestyle=':', color=color, linewidth=1.3)
    ax2.text(idx_n + 0.3, nivel + 1, f'{nivel}% ({idx_n+1} vars)', color=color, fontsize=9)

ax1.set_xticks(range(len(fi_sorted)))
ax1.set_xticklabels(fi_sorted['feature'], rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Importancia individual (%)')
ax2.set_ylabel('Importancia acumulada (%)')
ax1.set_title('Diagrama de Pareto — Importancia de Variables (Modelo Cox)',
              fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7. Distribución y Eficacia de las Alertas


In [ ]:
orden_niveles   = ['BAJO', 'MEDIO', 'ALTO']
colores_alerta  = {'BAJO': VERDE, 'MEDIO': AMARILLO, 'ALTO': ROJO}

nivel_counts = alertas['nivel_alerta'].value_counts().reindex(orden_niveles)
nivel_pct    = nivel_counts / nivel_counts.sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras
bars = axes[0].bar(
    nivel_counts.index, nivel_counts.values,
    color=[colores_alerta[n] for n in nivel_counts.index],
    width=0.5, edgecolor='white'
)
for bar, pct in zip(bars, nivel_pct):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Cantidad de créditos')
axes[0].set_title('Distribución de Niveles de Alerta')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Donut
axes[1].pie(nivel_counts.values,
            labels=nivel_counts.index,
            colors=[colores_alerta[n] for n in nivel_counts.index],
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'linewidth': 3, 'edgecolor': 'white'},
            pctdistance=0.75, textprops={'fontsize': 12})
axes[1].add_artist(plt.Circle((0,0), 0.45, fc='white'))
axes[1].set_title('Proporción de Alertas')

plt.suptitle('Sistema de Alertas SAT — Distribución por Nivel de Riesgo',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Probabilidades de mora por nivel de alerta
surv_cols = ['prob_mora_30d', 'prob_mora_60d', 'prob_mora_90d', 'atraso_30']
surv_cols = [c for c in surv_cols if c in alertas.columns]

mora_por_nivel = alertas.groupby('nivel_alerta')[surv_cols].agg(['mean','count']).reset_index()

# Tabla resumen
resumen_alertas = alertas.groupby('nivel_alerta').agg(
    n=('prob_mora_30d', 'count'),
    prob_mora_30d=('prob_mora_30d', 'mean'),
    tasa_mora_real=('atraso_30', 'mean'),
    tiempo_esperado=('tiempo_esperado_mora', 'mean'),
).reindex(orden_niveles).reset_index()
resumen_alertas['tasa_mora_real'] *= 100
resumen_alertas['prob_mora_30d']  *= 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tasa de mora real por nivel
bars = axes[0].bar(
    resumen_alertas['nivel_alerta'], resumen_alertas['tasa_mora_real'],
    color=[colores_alerta[n] for n in resumen_alertas['nivel_alerta']],
    width=0.5, edgecolor='white'
)
for bar, row in zip(bars, resumen_alertas.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{row.tasa_mora_real:.1f}%\n(n={row.n:,})',
                 ha='center', fontweight='bold', fontsize=10)
axes[0].set_ylabel('Tasa de mora real (atraso_30 observado, %)')
axes[0].set_title('Tasa de Mora Real por Nivel de Alerta', fontweight='bold')

# Tiempo esperado de mora por nivel
bars2 = axes[1].bar(
    resumen_alertas['nivel_alerta'], resumen_alertas['tiempo_esperado'],
    color=[colores_alerta[n] for n in resumen_alertas['nivel_alerta']],
    width=0.5, edgecolor='white'
)
for bar, row in zip(bars2, resumen_alertas.itertuples()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{row.tiempo_esperado:.2f} m',
                 ha='center', fontweight='bold', fontsize=10)
axes[1].set_ylabel('Tiempo esperado al incumplimiento (meses)')
axes[1].set_title('Tiempo Esperado de Mora por Nivel\n'
                  '(menor = más urgente)', fontweight='bold')

plt.suptitle('Efectividad de los Niveles de Alerta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

display(resumen_alertas.rename(columns={
    'nivel_alerta':    'Nivel',
    'n':               'N créditos',
    'prob_mora_30d':   'P(mora 30d) % media',
    'tasa_mora_real':  'Mora real % (atraso_30)',
    'tiempo_esperado': 'Tiempo esperado (m)',
}).style.format({'P(mora 30d) % media': '{:.1f}',
                 'Mora real % (atraso_30)': '{:.1f}',
                 'Tiempo esperado (m)': '{:.3f}'}).hide(axis='index'))


In [ ]:
# Composición de alertas por departamento — Top 7 por volumen
dept_nivel = alertas.groupby(
    ['nombre_departamento_particular', 'nivel_alerta']
).size().unstack(fill_value=0)
dept_nivel = dept_nivel.reindex(columns=orden_niveles, fill_value=0)
dept_nivel = dept_nivel.loc[dept_nivel.sum(axis=1).nlargest(7).index]
dept_pct   = dept_nivel.div(dept_nivel.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 6))
bottom = np.zeros(len(dept_pct))
for nivel, color in [('BAJO', VERDE), ('MEDIO', AMARILLO), ('ALTO', ROJO)]:
    if nivel in dept_pct.columns:
        vals = dept_pct[nivel].values
        ax.barh(dept_pct.index, vals, left=bottom,
                color=color, label=nivel, edgecolor='white')
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 8:
                ax.text(b + v/2, i, f'{v:.0f}%', ha='center', va='center',
                        fontsize=9, color='white', fontweight='bold')
        bottom += vals

ax.set_xlabel('Distribución de alertas (%)')
ax.set_title('Composición de Alertas por Departamento (Top 7 por volumen)',
             fontweight='bold')
ax.legend(title='Nivel')
ax.set_xlim([0, 100])
plt.tight_layout()
plt.show()


---
## 8. Curvas Kaplan-Meier por Nivel de Alerta

Una buena separación entre curvas KM de los tres niveles confirma que
el sistema de alertas discrimina efectivamente el riesgo de supervivencia.
El test log-rank verifica si las diferencias son estadísticamente significativas.

In [ ]:
# Asignar nivel de alerta a las observaciones del test set
nivel_test = np.where(prob_mora >= umbral_alto, 'ALTO',
             np.where(prob_mora >= umbral_medio, 'MEDIO', 'BAJO'))

fig, ax = plt.subplots(figsize=(10, 6))

for nivel, color in [('ALTO', ROJO), ('MEDIO', AMARILLO), ('BAJO', VERDE)]:
    mask = nivel_test == nivel
    n    = mask.sum()
    kmf  = KaplanMeierFitter()
    kmf.fit(yt_test[mask], ye_test[mask], label=f'{nivel} (n={n:,})')
    kmf.plot_survival_function(ax=ax, color=color, linewidth=2.5,
                               ci_show=True, ci_alpha=0.12)

# Test log-rank ALTO vs BAJO
mask_a = nivel_test == 'ALTO'
mask_b = nivel_test == 'BAJO'
lr = logrank_test(yt_test[mask_a], yt_test[mask_b],
                  ye_test[mask_a], ye_test[mask_b])

ax.set_title(f'Curvas Kaplan-Meier por Nivel de Alerta\n'
             f'(log-rank ALTO vs BAJO: p = {lr.p_value:.2e})',
             fontweight='bold')
ax.set_xlabel('Tiempo (meses)')
ax.set_ylabel('S(t) — P(no incumplir hasta t)')
ax.set_xlim([0, 40])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

# Tasa de incumplimiento por nivel (validación)
for nivel in ['ALTO', 'MEDIO', 'BAJO']:
    mask = nivel_test == nivel
    tasa = ye_test[mask].mean()
    print(f'{nivel:5s}: n={mask.sum():6,}  tasa_mora={tasa:.3f}  '
          f'prob_mora_media={prob_mora[mask].mean():.3f}')


---
## 9. Comparación Train vs Test — Sobreajuste


In [ ]:
# C-index por subconjunto y distribución de scores
c_test_val  = concordance_index(yt_test,  -risk_test,  ye_test)
c_train_val = concordance_index(yt_train, -risk_train, ye_train)
brecha      = c_train_val - c_test_val

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — Barras C-index —
vals_c   = [c_train_val, c_test_val]
labels_c = ['Train', 'Test']
colors_c = [AZUL, ROJO]
bars = axes[0].bar(labels_c, vals_c, color=colors_c, width=0.4, edgecolor='white')
for bar, val in zip(bars, vals_c):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=12)
axes[0].axhline(0.85, color='gray', linestyle='--', linewidth=1.3, label='Umbral bueno (0.85)')
axes[0].set_ylim([0.85, 1.0])
axes[0].set_ylabel('C-index de Harrell')
axes[0].set_title(f'C-index Train vs Test\nBrecha = {brecha:.4f}'
                  f'  {"✓ Sin sobreajuste" if abs(brecha) < 0.05 else "⚠ Posible sobreajuste"}',
                  fontweight='bold')
axes[0].legend()

# — Distribución del log(score) en train vs test —
for data, color, label in [
    (np.log(risk_train + 1e-9), AZUL, 'Train'),
    (np.log(risk_test  + 1e-9), ROJO, 'Test'),
]:
    axes[1].hist(data, bins=50, alpha=0.5, color=color, label=label, density=True)
    sns.kdeplot(data, ax=axes[1], color=color, linewidth=2)

axes[1].set_xlabel('log(partial hazard)')
axes[1].set_ylabel('Densidad')
axes[1].set_title('Distribución del Score de Riesgo: Train vs Test', fontweight='bold')
axes[1].legend()

plt.suptitle('Análisis de Generalización del Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'C-index Train: {c_train_val:.4f}')
print(f'C-index Test:  {c_test_val:.4f}')
print(f'Brecha:        {brecha:.4f}  →  {"Sin sobreajuste (OK)" if abs(brecha) < 0.05 else "Revisar sobreajuste"}')


---
## 10. Reporte Final


In [ ]:
# Resumen del modelo Cox (top 15 variables)
summary_top = cph.summary[['coef','exp(coef)','se(coef)','z','p',
                            'exp(coef) lower 95%','exp(coef) upper 95%']].head(15)
summary_top.columns = ['β','HR','SE(β)','z','p','HR CI lower','HR CI upper']

display(
    summary_top.style
    .format({'β': '{:.4f}', 'HR': '{:.4f}', 'SE(β)': '{:.4f}',
             'z': '{:.2f}', 'p': '{:.4f}',
             'HR CI lower': '{:.4f}', 'HR CI upper': '{:.4f}'})
    .background_gradient(cmap='RdBu_r', subset=['β'], vmin=-2, vmax=2)
    .set_caption('Coeficientes del Modelo Cox PH — Top 15 por |β|')
)


In [ ]:
# Resumen visual final — métricas del modelo de supervivencia
metricas_finales = {
    'C-index\n(test)':   c_test_val,
    'C-index\n(train)':  c_train_val,
    '1 - Brier\n(t=1)':  1 - metricas['brier_t1'],
    'S(1) mediana\n(1-mora)': metricas['s1_mediana_test'],
    'AUC-ROC\n(aprox)':  roc_auc_score(ye_test, prob_mora),
}

fig, ax = plt.subplots(figsize=(13, 5))
nombres = list(metricas_finales.keys())
valores = list(metricas_finales.values())

bar_cols = [ROJO if v < 0.80 else AMARILLO if v < 0.90 else VERDE for v in valores]
bars = ax.bar(nombres, valores, color=bar_cols, width=0.55, edgecolor='white')

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold', fontsize=12)

ax.axhline(0.80, color='gray',  linestyle='--', linewidth=1.2, alpha=0.7, label='Bueno (0.80)')
ax.axhline(0.90, color='green', linestyle='--', linewidth=1.2, alpha=0.7, label='Excelente (0.90)')
ax.set_ylim([0.75, 1.02])
ax.set_ylabel('Valor')
ax.set_title('Resumen de Métricas — Modelo SAT Crédito (Cox PH)',
             fontweight='bold', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

print('\n=== CONCLUSIÓN ===')
print(f'  C-index test:         {c_test_val:.4f}  → Discriminación excelente')
print(f'  Brecha overfitting:   {brecha:.4f}      → Sin sobreajuste')
print(f'  Brier Score (t=1):    {metricas["brier_t1"]:.4f}  → Calibración aceptable')
print(f'  Umbral ALTO:          {umbral_alto:.4f}  → P(mora 30d) ≥ este valor')
print(f'  Umbral MEDIO:         {umbral_medio:.4f}  → P(mora 30d) ≥ este valor')
print(f'\nEl modelo de Cox PH permite responder «¿cuándo entrará en mora?» no solo «¿entrará?».')
print('La salida S(t|X) da probabilidades calibradas en múltiples horizontes temporales.')
